# Laboration 3: Optimising a Catalytic Reaction

**Team name:**
**Members:**
**Date:**

Follow `instruction.md` for the full task description, timing, and the
Checkpoint questions. This notebook is your **working notebook** — light
comments are enough here; your answers to the Checkpoints go on a
**separate answer sheet**, not in this file.

Before you start: open `plant_client.py` and fill in `SERVER_URL`,
`TEAM`, and `TOKEN` (given to you by your instructor).

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import pyDOE3
from scipy.stats import qmc, t as t_dist
from scipy import stats
from scipy.optimize import minimize

from plant_client import run_experiment, leaderboard

RANDOM_STATE = 42

# Keep every experiment you run here -- you'll need the full history in
# Step 3, not just your latest batch.
history = pd.DataFrame(columns=["T", "c", "rpm", "rate"])

def log(result):
    """Append a run_experiment() result to the running history table."""
    global history
    history = pd.concat([history, result], ignore_index=True)
    return history


# Setup for Colab: download plant_client.py and fill in the server address,
# team name and token, so `from plant_client import ...` below works.
import importlib, re, sys, urllib.request

---
## Step 1 — Factorial Screening (0:00–0:45)

Three candidate factors this time: **c**, **T**, and **stirring rate
(rpm)** — nobody has confirmed rpm actually matters, that's what this
step is for.

- [ ] Design a 2³ factorial (8 corner points) around the OVAT point (c=2.0, T=25.0) and a sensible centre rpm, plus a few centre-point replicates
- [ ] Run it via `run_experiment`
- [ ] Code each factor to +/-1 (e.g. `c_coded = (c - 2.0) / 0.5`) before fitting -- see the TODO below for why this matters
- [ ] Fit `rate ~ c_coded * T_coded * rpm_coded` on the corner points — which effects are significant? (→ **Checkpoint 1a**)
- [ ] If rpm (and its interactions) aren't significant, drop it and fix it at a convenient value from here on
- [ ] Compare centre-point average to corner-point average — is there curvature? (→ **Checkpoint 1b**)

In [38]:
import numpy as np

# TODO:
# 1. choose your factor ranges (e.g. c +/- 0.5 mM, T +/- 5 degC around
# the OVAT point, rpm +/- 200 around a centre value like 500)
# 2. build the 8 corner points (save in a df)
# 3. Build a few centre-point repeats (save in a separate df)

# HINT: itertools.product(levels_c, levels_T, levels_rpm)
# is the easiest way to build the 2^3 corners.
n_reps = 2   # two replicates per design point
design_3 = pyDOE3.ff2n(3)
df_design = pd.DataFrame(design_3, columns=['T', 'c', 'rpm, rate'])
df_coded = df_design
print(df_coded)
# Add natural units
df_design['T'] = 25 + 5*df_design['T']
df_design['c'] = 2   +   0.5*df_design['c']
df_design['rpm'] = 500 + 200*df_design['rpm']
df_design['rate'] = 5.77

print(f'Total runs: {len(df_design)}  (2³ × {n_reps} replicates)')
print(df_design.head(8).to_string(index=False))

     T    c  rpm
0 -1.0 -1.0 -1.0
1 -1.0 -1.0  1.0
2 -1.0  1.0 -1.0
3 -1.0  1.0  1.0
4  1.0 -1.0 -1.0
5  1.0 -1.0  1.0
6  1.0  1.0 -1.0
7  1.0  1.0  1.0
Total runs: 8  (2³ × 2 replicates)
   T   c   rpm
20.0 1.5 300.0
20.0 1.5 700.0
20.0 2.5 300.0
20.0 2.5 700.0
30.0 1.5 300.0
30.0 1.5 700.0
30.0 2.5 300.0
30.0 2.5 700.0


In [24]:
# TODO:
# 1. run the design with result = run_experiment(name="...", T=..., c=..., rpm=...).
# You will have to run the corner points and centre-points
# separately (it is needed for the next parts of the lab)
# 2. Use log(result) to add it to `history`.

# TIP: the run_experiment function is defined in plant_client.py.

print("Running corner point experiments...")
for i, row in df_design.iterrows():
    # Wrap the scalar values in lists as run_experiment expects iterable inputs
    result = run_experiment(name=f"corner_point_{i+1}", T=[row['T']], c=[row['c']], rpm=[row['rpm']])
    log(result)
    print(f"Experiment {i+1} (Corner): T={row['T']}, c={row['c']}, rpm={row['rpm']} -> Rate={result['rate'].iloc[0]}")

# NOTE: The center points are currently not being run due to previous changes.
# If you need to run center points, you would need a similar loop for df_center.
# For now, I'm focusing on fixing the immediate error for df_design.

print("\nAll experiments for factorial screening completed and logged to history.")
display(history)

Running corner point experiments...
Team 'Labgrp17': 1 experiment(s) this call, 1 total so far.
Experiment 1 (Corner): T=20.0, c=1.5, rpm=300.0 -> Rate=3.3286


/tmp/ipykernel_692/4166362918.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  history = pd.concat([history, result], ignore_index=True)


Team 'Labgrp17': 1 experiment(s) this call, 2 total so far.
Experiment 2 (Corner): T=20.0, c=1.5, rpm=700.0 -> Rate=3.4916
Team 'Labgrp17': 1 experiment(s) this call, 3 total so far.
Experiment 3 (Corner): T=20.0, c=2.5, rpm=300.0 -> Rate=5.1698
Team 'Labgrp17': 1 experiment(s) this call, 4 total so far.
Experiment 4 (Corner): T=20.0, c=2.5, rpm=700.0 -> Rate=5.0403
Team 'Labgrp17': 1 experiment(s) this call, 5 total so far.
Experiment 5 (Corner): T=30.0, c=1.5, rpm=300.0 -> Rate=6.1168
Team 'Labgrp17': 1 experiment(s) this call, 6 total so far.
Experiment 6 (Corner): T=30.0, c=1.5, rpm=700.0 -> Rate=6.0688
Team 'Labgrp17': 1 experiment(s) this call, 7 total so far.
Experiment 7 (Corner): T=30.0, c=2.5, rpm=300.0 -> Rate=7.6068
Team 'Labgrp17': 1 experiment(s) this call, 8 total so far.
Experiment 8 (Corner): T=30.0, c=2.5, rpm=700.0 -> Rate=7.6411

All experiments for factorial screening completed and logged to history.


,T,c,rpm,rate
0,20.0,1.5,300.0,3.3286
1,20.0,1.5,700.0,3.4916
2,20.0,2.5,300.0,5.1698
3,20.0,2.5,700.0,5.0403
4,30.0,1.5,300.0,6.1168
5,30.0,1.5,700.0,6.0688
6,30.0,2.5,300.0,7.6068
7,30.0,2.5,700.0,7.6411


In [41]:

# TODO:
# 1. For the result, code each factor to +/-1 using the factorial's own half-ranges
# (e.g. c_coded = (c - 2.0) / 0.5) and add these as new columns.
# 2. THEN fit rate ~ c_coded * T_coded * rpm_coded (statsmodels.formula.api.ols)
# ONLY on the corner points. (Fitting on raw units (mM, degC, rpm) instead will give
# you a badly ill-conditioned model where every effect looks non-significant
# that's a red flag to code your variables, not a real finding.)

# TIP: .summary() shows you the coefficients, signs, and p-values for every
# main effect and interaction.
# Define center points and half-ranges for coding
center_c = 2.0
half_range_c = 0.5

center_T = 25.0
half_range_T = 5.0

center_rpm = 500.0
half_range_rpm = 200.0

# Create a DataFrame for the model, using a copy of history to add coded variables.
# The 'history' DataFrame contains the actual 'rate' values from the experiments.
df_model_data = history.copy()

# Code each factor to +/-1
df_model_data['c_coded'] = (df_model_data['c'] - center_c) / half_range_c
df_model_data['T_coded'] = (df_model_data['T'] - center_T) / half_range_T
df_model_data['rpm_coded'] = (df_model_data['rpm'] - center_rpm) / half_range_rpm

# Display the prepared data with coded factors and rate
print("Data for OLS model:")
display(df_model_data[['c_coded', 'T_coded', 'rpm_coded', 'rate']])

# Fit rate ~ c_coded * T_coded * rpm_coded (statsmodels.formula.api.ols)
# ONLY on the corner points (df_model_data currently contains only corner points based on previous steps).
model = smf.ols(
    'rate ~ c_coded * T_coded * rpm_coded',
    data=df_model_data
).fit()

# Print the model summary
print("\nOLS Model Summary:")
print(model.summary())


Data for OLS model:


,c_coded,T_coded,rpm_coded,rate
0,-1.0,-1.0,-1.0,3.3286
1,-1.0,-1.0,1.0,3.4916
2,1.0,-1.0,-1.0,5.1698
3,1.0,-1.0,1.0,5.0403
4,-1.0,1.0,-1.0,6.1168
5,-1.0,1.0,1.0,6.0688
6,1.0,1.0,-1.0,7.6068
7,1.0,1.0,1.0,7.6411



OLS Model Summary:
                            OLS Regression Results                            
Dep. Variable:                   rate   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Fri, 02 Oct 2026   Prob (F-statistic):                nan
Time:                        08:42:03   Log-Likelihood:                 262.34
No. Observations:                   8   AIC:                            -508.7
Df Residuals:                       0   BIC:                            -508.1
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
In

/usr/local/lib/python3.13/dist-packages/statsmodels/regression/linear_model.py:2032: RuntimeWarning: divide by zero encountered in divide
  np.divide(self.nobs - self.k_constant, self.df_resid) * (1 - self.rsquared)
/usr/local/lib/python3.13/dist-packages/statsmodels/regression/linear_model.py:2032: RuntimeWarning: invalid value encountered in scalar multiply
  np.divide(self.nobs - self.k_constant, self.df_resid) * (1 - self.rsquared)
/usr/local/lib/python3.13/dist-packages/statsmodels/regression/linear_model.py:1954: RuntimeWarning: divide by zero encountered in scalar divide
  return np.dot(wresid, wresid) / self.df_resid


Baserat på analysen av OLS-rapporten, om `rpm` och dess interaktioner bedöms vara statistiskt obetydliga (dvs. p-värdena är höga), kan vi förenkla modellen genom att ta bort `rpm` som en variabel.

När du har tagit bort `rpm` som en rörlig variabel från din modell kommer du att använda ett **fast `rpm`-värde** för alla framtida experiment. Detta värde väljs oftast som mittpunkten (t.ex. 500 rpm) från ditt ursprungliga designområde.

In [42]:
# TODO: decide whether to drop rpm based on the model above. If you do,
# note it here (and on the answer sheet, Checkpoint 1a) and pick a fixed      we drop the rpm based
# rpm value to use for the rest of the lab.

df_model_data_no_rpm = df_model_data.drop(columns=['rpm', 'rpm_coded'])
fixed_rpm_value = 500.0

print(f"'rpm' is Droped. Fixed rpm value:{fixed_rpm_value}")
print("De första raderna i den nya DataFrame utan rpm:")
display(df_model_data_no_rpm.head())

'rpm' is Droped. Fixed rpm value:500.0
De första raderna i den nya DataFrame utan rpm:


,T,c,rate,c_coded,T_coded
0,20.0,1.5,3.3286,-1.0,-1.0
1,20.0,1.5,3.4916,-1.0,-1.0
2,20.0,2.5,5.1698,1.0,-1.0
3,20.0,2.5,5.0403,1.0,-1.0
4,30.0,1.5,6.1168,-1.0,1.0


In [43]:
# TODO: compare the centre-point average rate to the average of the 8
# corner points. A meaningfully different value suggests curvature.
# Reason about this on the answer sheet (Checkpoint 1b), not just here.

df_centre_coded = pd.DataFrame({'T': [25, 25, 25], 'c': [2, 2, 2], 'rpm': [0, 0, 0]})


# --- Run experiments for curvature comparison ---
# Create temporary design dataframes with fixed rpm for the curvature comparison
df_design_for_curvature = df_design.copy()
df_design_for_curvature['rpm'] = fixed_rpm_value

df_center_for_curvature = df_centre_coded.copy()
df_center_for_curvature['rpm'] = fixed_rpm_value

print("Running corner point experiments with fixed rpm for curvature analysis...")
for i, row in df_design_for_curvature.iterrows():
    result = run_experiment(name=f"corner_point_curvature_{i+1}", T=[row['T']], c=[row['c']], rpm=[row['rpm']])
    result['point_set'] = 'curvature_corner' # Add a tag to identify these specific runs
    log(result)
    print(f"Experiment {i+1} (Corner, Fixed RPM): T={row['T']}, c={row['c']}, rpm={row['rpm']} -> Rate={result['rate'].iloc[0]}")

print("Running center point experiments with fixed rpm for curvature analysis...")
for i, row in df_center_for_curvature.iterrows():
    result = run_experiment(name=f"center_point_curvature_{i+1}", T=[row['T']], c=[row['c']], rpm=[row['rpm']])
    result['point_set'] = 'curvature_center' # Add a tag to identify these specific runs
    log(result)
    print(f"Experiment {i+1} (Center, Fixed RPM): T={row['T']}, c={row['c']}, rpm={row['rpm']} -> Rate={result['rate'].iloc[0]}")

print("\nAll experiments for curvature comparison completed and logged to history.")
display(history)

# Calculate average rates for corner points and center points from the 'curvature' set
# Filter history to only include the experiments run for this curvature analysis
curvature_experiments = history[history['point_set'].isin(['curvature_corner', 'curvature_center'])]

average_corner_rate = curvature_experiments[curvature_experiments['point_set'] == 'curvature_corner']['rate'].mean()
average_center_rate = curvature_experiments[curvature_experiments['point_set'] == 'curvature_center']['rate'].mean()

print(f"\nAverage rate for corner points (with fixed rpm): {average_corner_rate:.4f}")
print(f"Average rate for center points (with fixed rpm): {average_center_rate:.4f}")

# Compare and suggest curvature
# The threshold (0.1) is a heuristic; a statistical test might be used in a more rigorous setting.
if abs(average_center_rate - average_corner_rate) > 0.3:
    print("There is a meaningful difference between center and corner point average rates, suggesting curvature in the response surface.")
else:
    print("There is no meaningful difference between center and corner point average rates, suggesting a linear relationship.")

Running corner point experiments with fixed rpm for curvature analysis...
Team 'Labgrp17': 1 experiment(s) this call, 9 total so far.
Experiment 1 (Corner, Fixed RPM): T=20.0, c=1.5, rpm=500.0 -> Rate=3.4203
Team 'Labgrp17': 1 experiment(s) this call, 10 total so far.
Experiment 2 (Corner, Fixed RPM): T=20.0, c=1.5, rpm=500.0 -> Rate=3.6356
Team 'Labgrp17': 1 experiment(s) this call, 11 total so far.
Experiment 3 (Corner, Fixed RPM): T=20.0, c=2.5, rpm=500.0 -> Rate=4.8713
Team 'Labgrp17': 1 experiment(s) this call, 12 total so far.
Experiment 4 (Corner, Fixed RPM): T=20.0, c=2.5, rpm=500.0 -> Rate=5.0132
Team 'Labgrp17': 1 experiment(s) this call, 13 total so far.
Experiment 5 (Corner, Fixed RPM): T=30.0, c=1.5, rpm=500.0 -> Rate=6.2985
Team 'Labgrp17': 1 experiment(s) this call, 14 total so far.
Experiment 6 (Corner, Fixed RPM): T=30.0, c=1.5, rpm=500.0 -> Rate=6.076
Team 'Labgrp17': 1 experiment(s) this call, 15 total so far.
Experiment 7 (Corner, Fixed RPM): T=30.0, c=2.5, rpm=500.

,T,c,rpm,rate,point_set
0,20.0,1.5,300.0,3.3286,NaN
1,20.0,1.5,700.0,3.4916,NaN
2,20.0,2.5,300.0,5.1698,NaN
3,20.0,2.5,700.0,5.0403,NaN
4,30.0,1.5,300.0,6.1168,NaN
5,30.0,1.5,700.0,6.0688,NaN
6,30.0,2.5,300.0,7.6068,NaN
7,30.0,2.5,700.0,7.6411,NaN
8,20.0,1.5,500.0,3.4203,curvature_corner
9,20.0,1.5,500.0,3.6356,curvature_corner



Average rate for corner points (with fixed rpm): 5.5247
Average rate for center points (with fixed rpm): 5.6375
There is a meaningful difference between center and corner point average rates, suggesting curvature in the response surface.


---
## Step 2 — Steepest Ascent (0:45–1:30)

- [ ] Use the significant main effects from Step 1 (c and T, assuming rpm was dropped) to compute an ascent direction
- [ ] Run a sequence of points stepping along that direction, until the rate stops improving. If you fixed the repm value, keep it fixed at your chosen value.
- [ ] Keep `log()`-ing every result so `history` stays complete

In [50]:
# TODO:
# 1. Compute the steepest-ascent direction from your Step 1 model's c
# and T coefficients (the ratio of the fitted effects gives the relative
# step sizes)
# 2. Choose a step size and take your first step. If you fixed
# your rpm value, pass it on every call.
df_model_data = history.copy()

# Code each factor to +/-1
df_model_data['c_coded'] = (df_model_data['c'] - center_c) / half_range_c
df_model_data['T_coded'] = (df_model_data['T'] - center_T) / half_range_T
df_model_data['rpm_coded'] = (df_model_data['rpm'] - center_rpm) / half_range_rpm

# Fit rate ~ c_coded * T_coded * rpm_coded (statsmodels.formula.api.ols)
# ONLY on the corner points (df_model_data currently contains only corner points based on previous steps).
model = smf.ols(
    'rate ~ c_coded * T_coded * rpm_coded',
    data=df_model_data
).fit()
# Print the model summary
print("\nOLS Model Summary:")
print(model.summary())
b1, b2 = model.params['T_coded'], model.params['c_coded']
print(f'First-order coefficients: b1 = {b1:.3f}, b2 = {b2:.3f}')
print(f'Steepest ascent direction: ({b1:.3f}, {b2:.3f})')
#step size
step_T_coded = 0.2

#step size in coded
step_c_coded = step_T_coded * (b2 / b1)

# steps in natural units
step_T = step_T_coded * half_range_T
step_c = step_c_coded * half_range_c

print(f"Stegstorlek i fysikaliska enheter:")
print(f"  Delta T: +{step_T:.2f} °C per steg")
print(f"  Delta c: +{step_c:.3f} mM per steg\n")
for step in range(1, 6):
    # Beräkna nya värden för detta steg
    target_T = center_T + step_T
    target_c = center_c + step_c

result = run_experiment(
        name=f"steepest_ascent_step_{step}",
        T=[target_T],
        c=[target_c],
        rpm=[fixed_rpm_value]
    )
result['point_set'] = 'steepest_ascent'
log(result)
display(history)






OLS Model Summary:
                            OLS Regression Results                            
Dep. Variable:                   rate   R-squared:                       0.994
Model:                            OLS   Adj. R-squared:                  0.990
Method:                 Least Squares   F-statistic:                     245.5
Date:                Fri, 02 Oct 2026   Prob (F-statistic):           3.45e-11
Time:                        10:07:48   Log-Likelihood:                 15.072
No. Observations:                  19   AIC:                            -14.14
Df Residuals:                      11   BIC:                            -6.588
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
In

,T,c,rpm,rate,point_set
0,20.0,1.500000,300.0,3.3286,NaN
1,20.0,1.500000,700.0,3.4916,NaN
2,20.0,2.500000,300.0,5.1698,NaN
3,20.0,2.500000,700.0,5.0403,NaN
4,30.0,1.500000,300.0,6.1168,NaN
5,30.0,1.500000,700.0,6.0688,NaN
6,30.0,2.500000,300.0,7.6068,NaN
7,30.0,2.500000,700.0,7.6411,NaN
8,20.0,1.500000,500.0,3.4203,curvature_corner
9,20.0,1.500000,500.0,3.6356,curvature_corner


In [29]:
# TODO:
# Perform a steepest ascent along the ascent direction, starting from the OVAT point.
# Run the experiments using result = run_experiment(name, T, c, rpm)
# and log each result using log(result).
# Print the best (T, c, rate) you've found so far after each step.
# Continue the steepest ascent until the rate stops improving, using a reasonable criteria.
import numpy as np
import pandas as pd

# Utgångspunkt (mittpunkten för T och c i kodade koordinater)
x_start = np.array([0.0, 0.0]) # [T_coded, c_coded]

# Beräkna stegen i kodade enheter enligt din formel
step_vector_coded = (0.2 / abs(b1)) * np.array([b1, b2])

# Hämta den bästa hittills observerade reaktionshastigheten som baslinje
best_rate = history['rate'].max()
print(f"Utgångspunktens bästa rate: {best_rate:.4f} mol/s\n")

# Skapa sökvägen (path) för t.ex. upp till 10 steg
max_steps = 10
consecutive_drops = 0

for s in range(1, max_steps + 1):
    # Beräkna kodad punkt på söklinjen
    current_coded = x_start + s * step_vector_coded

    # Konvertera tillbaka till naturliga (fysikaliska) enheter
    target_T = center_T + current_coded[0] * half_range_T
    target_c = center_c + current_coded[1] * half_range_c

    print(f"--- Testar steg {s} ---")
    print(f"Inställningar: T = {target_T:.2f} °C, c = {target_c:.3f} mM (rpm = {fixed_rpm_value})")

    # Kör experimentet
    result = run_experiment(
        name=f"steepest_ascent_step_{s}",
        T=[target_T],
        c=[target_c],
        rpm=[fixed_rpm_value]
    )
    result['point_set'] = 'steepest_ascent'
    log(result)

    current_rate = result['rate'].iloc[0]
    print(f"Resultat: Rate = {current_rate:.4f} mol/s")

    # Kontrollera utvecklingen
    if current_rate > best_rate:
        best_rate = current_rate
        consecutive_drops = 0  # Återställ räknaren eftersom vi nådde en ny topp
        print(f"Ny bästa rate hittad: {best_rate:.4f} mol/s!")
    else:
        consecutive_drops += 1
        print(f"Värdet gick ner/stagnerade jämfört med bästa ({best_rate:.4f}). Antal nedgångar i rad: {consecutive_drops}")

    # Avbryt om värdet har gått ner två gånger i rad
    if consecutive_drops >= 2:
        print("\nAvbryter stigningen eftersom värdet har gått ner två gånger i rad.")
        break

print("\n--- Steepest Ascent Avslutad ---")
display(history.tail(12))

---
## Step 3 — Response Surface Optimisation (1:30–3:00)

- [ ] Design a Latin-hypercube set of points around the best region found in Step 2 (`scipy.stats.qmc.LatinHypercube`), rpm still fixed
- [ ] Run it, fit a quadratic model (`rate ~ c + T + I(c**2) + I(T**2) + c:T`)
- [ ] Contour plot of predicted rate, with your best point and the model's predicted optimum marked (→ **Checkpoint 3b**)
- [ ] Confirm the predicted optimum with 1-2 extra experiments (an informal first check -- Step 4 makes this rigorous)

In [30]:
# TODO: use qmc.LatinHypercube(d=2, seed=RANDOM_STATE) to generate sample
# points in [0,1]^2 for (c, T), then scale them to your chosen region.
# qmc.scale(sample, l_bounds, u_bounds) does the scaling for you. If you
# fixed your rpm value, rpm stays fixed at your chosen value for every point.

In [31]:
# TODO:
# 1. run the LHS design and log() the results.
# 2. Fit the quadratic model on the FULL history (df_full = history) or
# just the RSM-region points: your call, but say which on the answer sheet.

In [32]:
# TODO:
# 1. Build a grid over your explored (c, T) region. You will need
# to decide on the factor ranges, and the number of grid points.
# 2. Predict the rate with your fitted model at every grid point.
# 3. Make a contour plot using plt.contourf / ax.contour.
# Mark your best experimental point and the model's predicted optimum
# (e.g. from scipy.optimize.minimize on the negative of
# your fitted model, or just the grid's argmax).

In [33]:
# TODO:
# 1. Store the predicted optimum (c_star, T_star, rate_star)
# -- you'll reuse it in Step 4.
# 2. Run 1-2 confirmation experiments at (or near) your predicted optimum
# and compare the observed rate to the model's prediction.

---
## Step 4 — How Sure Are You? A Confidence Interval (3:00–3:45)

- [ ] Run 8-10 replicate experiments at your predicted optimum (same T*, c*, and rpm every time).
- [ ] Compute the sample mean and standard deviation of the resulting rates
- [ ] Build a 95% confidence interval on the true mean rate there (→ **Checkpoint 4a**)
- [ ] Check whether the OVAT baseline (5.77 mol/s) and your model's predicted rate (Checkpoint 3b) fall inside or outside your CI
- [ ] Reflect on the sample-size / CI-width trade-off (→ **Checkpoint 4b**)

In [34]:
# TODO: run 8-10 replicate experiments at (c_star, T_star), same settings
# every call. log() the results.

In [35]:
# TODO: compute mean, standard deviation (ddof=1), and a 95% confidence
# interval on the mean rate. Use scipy.stats.t.ppf(0.975, df=n-1) for the
# critical value -- same one-sample CI construction as Part III.
# Then check: does 5.77 (OVAT) fall inside your CI? Does your Step 3
# predicted rate (rate_star) fall inside your CI?

In [36]:
# TODO (Checkpoint 4b): using your own n and CI half-width, work out
# roughly how many replicates you'd need to halve the half-width
# (half-width shrinks with 1/sqrt(n)). Is that trade worth it given the
# contest rewards fewer experiments? Write your reasoning on the answer
# sheet, not just the number here.

---
## Wrap-up (3:45–4:00)

- [ ] Check your final experiment count and best rate with `leaderboard()`
- [ ] Compute your % improvement over the OVAT baseline of 5.77 mol/s
- [ ] Finalise your answer sheet (Checkpoints 1a–4c)

In [37]:
# TODO: leaderboard() and a final summary print of best rate found,
# total experiments used, and % improvement over 5.77 mol/s.